# Anonimización del dataset de Gout

Fuentes (carpeta `Dataset/`, hoja `DATOS` en todos los archivos):
- **Pedidos por sucursal:** `PEDIDO_PASTELERIA_[SUCURSAL]_[AÑO].xlsx` → FECHA, PRODUCTO, FECHA_ELAB, FECHA_VTO, PEDIDO, STOCK, DEVOLUCIONES, TOTAL
- **Resumen:** `PEDIDO_PASTELERIA_RESUMEN_[AÑO].xlsx` → una columna por sucursal + RESUMEN, STOCK_PLANTA, PROD_SEMANAL, DIF_PEDIDO, DIF_STOCK_PROD

Salidas:
- `data/pedidos_anon.csv`, `data/resumen_anon.csv` (en Drive, `MyDrive/gout/data/`) → para el repo
- `private/` → mapeos, **no subir** (agregar al `.gitignore`)

In [ ]:
from pathlib import Path
import re
import numpy as np
import pandas as pd

## Montar Drive

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

## Configuración

In [ ]:
DATA_DIR = Path("/content/drive/MyDrive/gout/dataset")
N_SKUS = 20               # tamaño de la muestra
MIN_DIAS_CON_VENTA = 0.5  # fracción mínima de (fecha, sucursal) con TOTAL > 0
ESCALA_CANTIDAD = None    # ej. 1.37 para ocultar volúmenes reales; None = sin escalar
SEED = 42
OUT_DATA = DATA_DIR.parent / "data"      # salidas en Drive, al lado de Dataset/
OUT_PRIV = DATA_DIR.parent / "private"

SUCURSALES = ["QUILMES", "MONTEVIDEO", "LACROZE", "JUNCAL", "LOMITAS",
              "GONTRAN_CHERRIER", "PANERA_ROSA", "PALACIO_DUHAU", "GOLDEN_CENTER"]
COLS_CANT_PED = ["PEDIDO", "STOCK", "DEVOLUCIONES", "TOTAL"]
COLS_CANT_RES = ["STOCK_PLANTA", "PROD_SEMANAL"]

rng = np.random.default_rng(SEED)

## 1. Carga

`PRODUCTO` no es clave única: cada día hay 101 filas pero 96 nombres (ECLAIR aparece 4 veces y PAN 3, son variantes distintas con el mismo nombre). El orden de filas es idéntico todos los días, así que se arma `SKU_KEY` = nombre + número de ocurrencia dentro del día.

In [ ]:
def agregar_sku_key(df, grupo):
    n = df.groupby(grupo + ["PRODUCTO"]).cumcount() + 1
    return df.assign(SKU_KEY=df["PRODUCTO"] + "#" + n.astype(str))

archivos = sorted(DATA_DIR.glob("PEDIDO_PASTELERIA_*.xlsx"))
if not archivos:  # buscar la carpeta en MyDrive si la ruta no es la correcta
    encontrados = sorted(Path("/content/drive/MyDrive").rglob("PEDIDO_PASTELERIA_*.xlsx"))
    carpetas = sorted({f.parent for f in encontrados})
    assert carpetas, "No se encontraron archivos PEDIDO_PASTELERIA_*.xlsx en MyDrive"
    assert len(carpetas) == 1, f"Hay varias carpetas candidatas, fijar DATA_DIR: {carpetas}"
    DATA_DIR = carpetas[0]
    OUT_DATA, OUT_PRIV = DATA_DIR.parent / "data", DATA_DIR.parent / "private"
    archivos = sorted(DATA_DIR.glob("PEDIDO_PASTELERIA_*.xlsx"))
print(f"Leyendo {len(archivos)} archivos de {DATA_DIR}")

patron = re.compile(r"PEDIDO_PASTELERIA_(.+)_(\d{4})\.xlsx")
pedidos, resumen = [], []
for f in archivos:
    suc, anio = patron.match(f.name).groups()
    d = pd.read_excel(f, sheet_name="DATOS")
    if suc == "RESUMEN":
        resumen.append(d)
    else:
        pedidos.append(d.assign(SUCURSAL=suc))

ped = agregar_sku_key(pd.concat(pedidos, ignore_index=True), ["SUCURSAL", "FECHA"])
res = agregar_sku_key(pd.concat(resumen, ignore_index=True), ["FECHA"])
for df in (ped, res):
    for c in ["FECHA", "FECHA_ELAB", "FECHA_VTO"]:
        df[c] = pd.to_datetime(df[c])

assert set(ped["SUCURSAL"]) == set(SUCURSALES)
print(f"Pedidos: {ped.shape} | SKUs: {ped['SKU_KEY'].nunique()} | nombres: {ped['PRODUCTO'].nunique()}")
print(f"Resumen: {res.shape}")

## 2. Sucursales
Permutación aleatoria → S01..S09 (no revela ranking ni el orden original).

In [ ]:
perm = rng.permutation(SUCURSALES)
map_suc = {s: f"S{i+1:02d}" for i, s in enumerate(perm)}
map_suc

## 3. SKUs elegibles
Cada SKU tiene fila todos los días (con ceros), así que la cobertura se mide como fracción de registros con `TOTAL > 0`.

In [ ]:
cobertura = ped.groupby("SKU_KEY")["TOTAL"].apply(lambda x: (x > 0).mean())
elegibles = cobertura[cobertura >= MIN_DIAS_CON_VENTA].index
print(f"Elegibles: {len(elegibles)} de {ped['SKU_KEY'].nunique()}")

## 4. Muestra estratificada por volumen (alto / medio / bajo)

In [ ]:
vol = ped[ped["SKU_KEY"].isin(elegibles)].groupby("SKU_KEY")["TOTAL"].sum()
tercil = pd.qcut(vol.rank(method="first"), 3, labels=["bajo", "medio", "alto"])
por_tercil = [N_SKUS // 3 + (1 if i < N_SKUS % 3 else 0) for i in range(3)]

muestra = []
for t, n in zip(["alto", "medio", "bajo"], por_tercil):
    candidatos = np.array(tercil[tercil == t].index, dtype=object)
    muestra += list(rng.choice(candidatos, size=min(n, len(candidatos)), replace=False))

muestra = list(rng.permutation(np.array(muestra, dtype=object)))
map_sku = {s: f"P{i+1:03d}" for i, s in enumerate(muestra)}
tercil[muestra].value_counts()

## 5. Pedidos por sucursal anonimizados

In [ ]:
ped_anon = (ped[ped["SKU_KEY"].isin(muestra)]
            .assign(SUCURSAL=lambda d: d["SUCURSAL"].map(map_suc),
                    SKU=lambda d: d["SKU_KEY"].map(map_sku))
            .drop(columns=["PRODUCTO", "SKU_KEY"]))
if ESCALA_CANTIDAD:
    ped_anon[COLS_CANT_PED] = (ped_anon[COLS_CANT_PED] * ESCALA_CANTIDAD).round().astype(int)

ped_anon = (ped_anon[["FECHA", "SUCURSAL", "SKU", "FECHA_ELAB", "FECHA_VTO"] + COLS_CANT_PED]
            .sort_values(["SUCURSAL", "SKU", "FECHA"]).reset_index(drop=True))
ped_anon.head()

## 6. Resumen anonimizado
Las columnas de sucursal se renombran y se **reordenan** S01..S09 (el orden original también identifica). Si se escala, `RESUMEN`, `DIF_PEDIDO` y `DIF_STOCK_PROD` se recalculan para mantener sus identidades (RESUMEN = Σ sucursales; DIF_PEDIDO = STOCK_PLANTA − RESUMEN; DIF_STOCK_PROD = STOCK_PLANTA − PROD_SEMANAL).

In [ ]:
res_anon = (res[res["SKU_KEY"].isin(muestra)]
            .assign(SKU=lambda d: d["SKU_KEY"].map(map_sku))
            .drop(columns=["PRODUCTO", "SKU_KEY"])
            .rename(columns=map_suc))
cols_suc = sorted(map_suc.values())

if ESCALA_CANTIDAD:
    res_anon[cols_suc + COLS_CANT_RES] = (res_anon[cols_suc + COLS_CANT_RES] * ESCALA_CANTIDAD).round().astype(int)
    res_anon["RESUMEN"] = res_anon[cols_suc].sum(axis=1)
    res_anon["DIF_PEDIDO"] = res_anon["STOCK_PLANTA"] - res_anon["RESUMEN"]
    res_anon["DIF_STOCK_PROD"] = res_anon["STOCK_PLANTA"] - res_anon["PROD_SEMANAL"]

res_anon = (res_anon[["FECHA", "SKU", "FECHA_ELAB", "FECHA_VTO"] + cols_suc +
                     ["RESUMEN", "STOCK_PLANTA", "PROD_SEMANAL", "DIF_PEDIDO", "DIF_STOCK_PROD"]]
            .sort_values(["SKU", "FECHA"]).reset_index(drop=True))
res_anon.head()

## 7. Verificaciones

In [ ]:
# Sin nombres originales en las salidas
assert not set(SUCURSALES) & (set(res_anon.columns) | set(ped_anon["SUCURSAL"]))
assert "PRODUCTO" not in ped_anon.columns and "PRODUCTO" not in res_anon.columns
assert set(ped_anon["SKU"]) == set(res_anon["SKU"]) == set(map_sku.values())

# Identidad del resumen
assert (res_anon[cols_suc].sum(axis=1) == res_anon["RESUMEN"]).all()

print(f"Pedidos:  {len(ped):,} -> {len(ped_anon):,} filas")
print(f"Resumen:  {len(res):,} -> {len(res_anon):,} filas")
print(ped_anon.groupby("SUCURSAL")["SKU"].nunique().rename("skus_por_sucursal").to_string())

## 8. Guardar

In [ ]:
OUT_DATA.mkdir(parents=True, exist_ok=True)
OUT_PRIV.mkdir(parents=True, exist_ok=True)

ped_anon.to_csv(OUT_DATA / "pedidos_anon.csv", index=False)
res_anon.to_csv(OUT_DATA / "resumen_anon.csv", index=False)

pd.DataFrame({"original": list(map_suc), "anon": list(map_suc.values())}) \
  .to_csv(OUT_PRIV / "mapeo_sucursales.csv", index=False)
pd.DataFrame({"original": [k.split("#")[0] for k in map_sku],
              "ocurrencia": [int(k.split("#")[1]) for k in map_sku],
              "anon": list(map_sku.values()),
              "tercil": [tercil[k] for k in map_sku]}) \
  .to_csv(OUT_PRIV / "mapeo_skus.csv", index=False)
print("Listo")